# Financial Market Regime & Event Intelligence Engine
## Notebook 06: Rule-Based Financial Event Classification

Welcome to Notebook 06! Building upon our news ingestion pipeline (Notebook 04) and FinBERT sentiment inference (Notebook 05), we now add a **transparent rule-based Event Classification layer** to categorize financial headlines into macro event categories.

### Objectives:
1. **Reproducible Pipeline**: Recreate news ingestion and FinBERT sentiment inference across S&P 500 benchmarks and key market-moving equities.
2. **Category Taxonomy**: Define 9 distinct event categories (`Monetary Policy`, `Inflation`, `Employment / Labor`, `Earnings`, `M&A / Corporate Action`, `Geopolitical`, `Commodities`, `Market / Index`, `Other`).
3. **Transparent Classifier**: Build a regex keyword-matching classifier that outputs both `event_type` and the specific `event_trigger` pattern matched.
4. **Documented Priority Hierarchy**: Enforce a strict priority order for headlines matching multiple keyword patterns.
5. **Data Inspection**: Display 15 classified records with headlines, assigned event types, triggers, sentiment labels, and confidence scores.
6. **Statistical & Cross-Tabulation Analysis**: Compute event distribution counts and cross-tabulate `event_type` against `sentiment_label`.
7. **Interactive Plotly Visualizations**: Plot event category frequencies and temporal timeline distributions.
8. **Case Studies & Limitations**: Inspect specific headline classifications and document modeling limitations.

---
### Step 1: Classification Methodology & Category Taxonomy

#### Why a Rule-Based Classifier?
- At this stage of the project, we do not have a human-labeled financial event training dataset. Using an uninterpretable black-box model without training data is unreliable.
- A **transparent rule-based classifier** using regular expression pattern matching offers complete interpretability, deterministic behavior, and clear auditability.

#### Documented Priority Hierarchy:
When a headline contains terms belonging to multiple categories (e.g., *"Fed cuts rates as inflation cools"*), the classifier evaluates categories in the following strict priority order to prevent ambiguous assignments:
1. `Monetary Policy` (Central bank actions, interest rates, FOMC)
2. `Inflation` (CPI, PPI, consumer prices)
3. `Employment / Labor` (Jobs, payrolls, unemployment)
4. `Earnings` (Corporate revenue, quarterly results, EPS, guidance)
5. `M&A / Corporate Action` (Acquisitions, mergers, buyouts, spin-offs)
6. `Geopolitical` (War, tariffs, trade sanctions, conflicts)
7. `Commodities` (Crude oil, gold, natural gas, energy)
8. `Market / Index` (S&P 500, Nasdaq, Dow, stock rallies/selloffs)
9. `Other` (Fallback category when no keyword pattern matches)

---
### Step 2: Import Required Libraries

**Why we do this:**
- `pandas` & `numpy`: Tabular processing and cross-tabulations.
- `plotly.express`: Interactive event category and timeline charts.
- `re`: Regular expressions for headline pattern matching.
- `yfinance`: News ingestion.
- `transformers`: FinBERT sentiment inference.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import yfinance as yf
import re
import torch
from transformers import pipeline

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 90)
print("Libraries successfully imported!")

Libraries successfully imported!


---
### Step 3: Ingest News & Run FinBERT Sentiment Pipeline

We recreate the news data ingestion across 12 tickers and apply pretrained FinBERT sentiment inference.

In [2]:
target_tickers = [
    "^GSPC", "SPY", "^VIX", "TLT", "GLD", "QQQ",
    "AAPL", "MSFT", "NVDA", "AMZN", "JPM", "GS"
]

raw_records = []
for ticker in target_tickers:
    try:
        news_items = yf.Ticker(ticker).news
        for item in news_items:
            content = item.get("content", item)
            headline = content.get("title")
            pub_date = content.get("pubDate")
            summary = content.get("summary") or content.get("description", "")
            
            provider_info = content.get("provider", {})
            publisher = provider_info.get("displayName", "Unknown") if isinstance(provider_info, dict) else "Unknown"
            
            canonical_info = content.get("canonicalUrl", {})
            click_info = content.get("clickThroughUrl", {})
            url = canonical_info.get("url") if isinstance(canonical_info, dict) and canonical_info.get("url") else (click_info.get("url", "") if isinstance(click_info, dict) else "")
            
            raw_records.append({
                "query_ticker": ticker,
                "headline": headline,
                "pub_date_raw": pub_date,
                "summary": summary,
                "publisher": publisher,
                "url": url
            })
    except Exception as e:
        print(f"Warning for {ticker}: {e}")

news_df = pd.DataFrame(raw_records)
news_df["published_at"] = pd.to_datetime(news_df["pub_date_raw"], utc=True)
news_df = news_df.dropna(subset=["headline"]).drop_duplicates(subset=["headline"]).copy()
news_df["summary"] = news_df["summary"].fillna("N/A")
news_df["publisher"] = news_df["publisher"].fillna("Unknown")
news_df["url"] = news_df["url"].fillna("")

# FinBERT Sentiment Inference
print("Running FinBERT sentiment inference...")
sentiment_pipeline = pipeline("sentiment-analysis", model="ProsusAI/finbert", tokenizer="ProsusAI/finbert")
nlp_results = sentiment_pipeline(news_df["headline"].tolist())

news_df["sentiment_label"] = [r["label"] for r in nlp_results]
news_df["sentiment_score"] = [round(r["score"], 4) for r in nlp_results]

print(f"Ingestion & Sentiment Pipeline complete. Total dataset shape: {news_df.shape}")

Running FinBERT sentiment inference...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Ingestion & Sentiment Pipeline complete. Total dataset shape: (108, 9)


---
### Step 4: Define Keyword Dictionaries & Implement Rule-Based Event Classifier

We build regex pattern rules for each category in accordance with our documented priority hierarchy.

In [3]:
# Define Event Rule Dictionaries ordered by Priority
EVENT_RULES = [
    ("Monetary Policy", [
        r"\bfed\b", r"\bfederal reserve\b", r"\binterest rate(s)?\b", r"\brate cut(s)?\b", 
        r"\brate hike(s)?\b", r"\bfomc\b", r"\bmonetary policy\b", r"\bcentral bank(s)?\b"
    ]),
    ("Inflation", [
        r"\binflation\b", r"\bcpi\b", r"\bppi\b", r"\bconsumer price(s)?\b", r"\bproducer price(s)?\b"
    ]),
    ("Employment / Labor", [
        r"\bjob(s)?\b", r"\bpayroll(s)?\b", r"\bunemployment\b", r"\bemployment\b", r"\bnonfarm\b", r"\blabor\b"
    ]),
    ("Earnings", [
        r"\bearning(s)?\b", r"\brevenue(s)?\b", r"\bprofit(s)?\b", r"\bquarterly result(s)?\b", r"\bguidance\b", r"\beps\b"
    ]),
    ("M&A / Corporate Action", [
        r"\bacquisition(s)?\b", r"\bmerger(s)?\b", r"\btakeover(s)?\b", r"\bbuyout(s)?\b", r"\bspin-off(s)?\b", r"\bdeal(s)?\b"
    ]),
    ("Geopolitical", [
        r"\bwar\b", r"\bsanction(s)?\b", r"\btariff(s)?\b", r"\btrade tension(s)?\b", r"\bconflict(s)?\b", r"\bgeopoliti\w*"
    ]),
    ("Commodities", [
        r"\boil\b", r"\bcrude\b", r"\bgold\b", r"\bnatural gas\b", r"\bcommodity\b", r"\bbrent\b"
    ]),
    ("Market / Index", [
        r"\bs&p 500\b", r"\bs&p\b", r"\bnasdaq\b", r"\bdow\b", r"\bstock(s)?\b", r"\bequity\b", r"\bequities\b", 
        r"\bmarket rally\b", r"\bmarket selloff\b", r"\bwall street\b"
    ])
]

def classify_headline_event(headline):
    if not isinstance(headline, str):
        return "Other", "N/A"
    
    headline_clean = headline.lower().strip()
    
    for category, patterns in EVENT_RULES:
        for pattern in patterns:
            match = re.search(pattern, headline_clean)
            if match:
                return category, match.group(0)
                
    return "Other", "N/A"

# Apply classifier to dataset
classified_results = [classify_headline_event(h) for h in news_df["headline"]]
news_df["event_type"] = [res[0] for res in classified_results]
news_df["event_trigger"] = [res[1] for res in classified_results]

print("=== Event Classification Complete ===\n")
print("--- Sample of 15 Classified News Headlines ---")
display_cols = ["headline", "event_type", "event_trigger", "sentiment_label", "sentiment_score"]
display(news_df[display_cols].head(15))

=== Event Classification Complete ===

--- Sample of 15 Classified News Headlines ---


,headline,event_type,event_trigger,sentiment_label,sentiment_score
0,Why Ed Yardeni is cutting his year-end target,Other,N/A,negative,0.6051
1,"Stock market today: Dow, S&P 500 post weekly losses as 10-year Treasury yield hovers n...",Market / Index,s&p 500,negative,0.9692
2,What S&P 500 Gains of 9.5% in the First Half Signal for the Rest of the Year,Market / Index,s&p 500,positive,0.8831
3,"If There's Just 1 Move All Investors Should Make Right Now, History Says It's This",Other,N/A,neutral,0.9329
4,The Stock Market's Biggest Companies Are Losing Their Grip. Here's the ETF I'd Buy If ...,Market / Index,stock,neutral,0.6156
5,Anthropic's IPO Is Coming. Here's What That Means for S&P 500 Investors.,Market / Index,s&p 500,neutral,0.9321
6,"Bank of America Sets 12-Month S&P 500 Target at 7,800, Sees 2% Upside",Market / Index,s&p 500,positive,0.5663
7,"Dow Jones Futures: Will The Market Rally Take Flight? Robinhood, Sandisk, AMD, Moderna...",Market / Index,dow,neutral,0.4948
8,History Says This Investment Strategy Can Build a $1 Million Portfolio. Here's How.,Other,N/A,neutral,0.8799
9,What History Reveals About Investing Through a Stock Market Crash,Market / Index,stock,neutral,0.9328


---
### Step 5: Event-Type Distribution & Sentiment Cross-Tabulation

We compute total counts per event category and cross-tabulate `event_type` against `sentiment_label`.

In [4]:
# 1. Event Type Distribution Table
event_counts = news_df["event_type"].value_counts()
event_pcts = (news_df["event_type"].value_counts(normalize=True) * 100).round(2)
event_dist_df = pd.DataFrame({
    "Event_Type": event_counts.index,
    "Article_Count": event_counts.values,
    "Percentage": event_pcts.values
})

print("--- Event-Type Distribution Table ---")
display(event_dist_df)

# 2. Cross-Tabulation: Event Type vs. Sentiment Label
print("\n--- Cross-Tabulation: Event Type vs. Sentiment Label ---")
crosstab_df = pd.crosstab(news_df["event_type"], news_df["sentiment_label"], margins=True)
display(crosstab_df)

--- Event-Type Distribution Table ---


,Event_Type,Article_Count,Percentage
0,Other,48,44.44
1,Market / Index,34,31.48
2,Monetary Policy,12,11.11
3,Commodities,8,7.41
4,Earnings,3,2.78
5,Employment / Labor,2,1.85
6,Inflation,1,0.93



--- Cross-Tabulation: Event Type vs. Sentiment Label ---


sentiment_label,negative,neutral,positive,All
event_type,,,,
Commodities,3,3,2,8
Earnings,2,1,0,3
Employment / Labor,2,0,0,2
Inflation,1,0,0,1
Market / Index,12,16,6,34
Monetary Policy,5,4,3,12
Other,12,30,6,48
All,37,54,17,108


---
### Step 6: Visualize Event Category Frequencies & Timeline (Plotly)

We plot event category volume and a stacked timeline of events over time.

In [5]:
# Plot 1: Event Categories Bar Chart
fig_events_bar = px.bar(
    event_dist_df,
    x="Event_Type",
    y="Article_Count",
    title="Financial Event Categories Frequency Distribution",
    labels={"Event_Type": "Event Category", "Article_Count": "Number of Articles"},
    color="Event_Type",
    template="plotly_white"
)
fig_events_bar.update_layout(title_x=0.5, showlegend=False)
fig_events_bar.show()

# Plot 2: Event Timeline Chart
timeline_df = news_df.copy()
timeline_df["date_only"] = timeline_df["published_at"].dt.date
daily_events_df = timeline_df.groupby(["date_only", "event_type"]).size().reset_index(name="count")

fig_events_ts = px.bar(
    daily_events_df,
    x="date_only",
    y="count",
    color="event_type",
    title="Financial Event Categories Over Time",
    labels={"date_only": "Publication Date", "count": "Article Count", "event_type": "Event Category"},
    template="plotly_white"
)
fig_events_ts.update_layout(title_x=0.5)
fig_events_ts.show()

---
### Step 7: Manual Headline Classification Case Studies

Below are sample headlines from our dataset illustrating how the regular expression rules assign `event_type` and capture `event_trigger`:

In [6]:
# Select 4 diverse classified headlines
sample_events = news_df.drop_duplicates(subset=["event_type"]).head(4)
display(sample_events[["headline", "event_type", "event_trigger", "sentiment_label", "publisher"]])

,headline,event_type,event_trigger,sentiment_label,publisher
0,Why Ed Yardeni is cutting his year-end target,Other,N/A,negative,Yahoo Finance Video
1,"Stock market today: Dow, S&P 500 post weekly losses as 10-year Treasury yield hovers n...",Market / Index,s&p 500,negative,Yahoo Finance
12,"Dow Drops To Record Worst Week In Six Months Amid Elevated Yields, Oil — NVDA, TSLA, S...",Commodities,oil,negative,Stocktwits
20,Options traders buy AI chip calls after Fed rate hike,Monetary Policy,fed,negative,Quartz


#### Classification Mechanics Analysis:
- **Headline with Central Bank Reference**: A headline containing *"Fed"* or *"interest rates"* matches the highest-priority `Monetary Policy` rule and extracts `event_trigger='fed'`.
- **Headline with Price Metrics**: A headline containing *"inflation"* or *"CPI"* matches `Inflation` with `event_trigger='inflation'`.
- **Headline with Market Movements**: A headline referring to *"S&P 500"* or *"stock rally"* matches `Market / Index` with `event_trigger='s&p 500'`.
- **Unmatched Headlines**: Headlines lacking explicit rule keywords are assigned to the `Other` category with `event_trigger='N/A'`.

---
### Step 8: Limitations of Rule-Based Event Classification

1. **Implicit Event Omissions**: Keyword matching fails to catch subtle or implicitly phrased events (e.g., *"Chairman Powell hints at upcoming policy adjustment"* might miss if explicit keywords like *'rate cut'* are absent).
2. **Multi-Event Headlines**: Headlines often touch upon multiple themes (e.g., *"Inflation spikes as crude oil surges"*). While the priority hierarchy assigns a single deterministic category (`Inflation`), secondary event context is lost.
3. **Lack of Deep Semantic Comprehension**: Regex matching operates on literal character strings rather than semantic contextual embeddings.
4. **Category Overlap**: Categories like `Market / Index` and `Earnings` frequently overlap in market news reporting.
5. **Sample Size Constraints**: The current sample reflects recent live news feeds.
6. **Requirement for Supervised ML**: To train a true supervised event classifier (e.g., a multi-label transformer), a human-annotated financial event benchmark dataset is required.